# 06 - Extracción NLP desde `chiefcomplaint`

Este notebook recoge la parte de extracción de variables a partir del texto breve de triaje (`chiefcomplaint`). 

La selección del modelo no se hace en este notebook. Esa comparación queda para `07_model_training.ipynb`.

La salida más importante es la representación **Bio_ClinicalBERT + SVD**, que fue la línea con más recorrido para el modelo final. Las variables LLM se conservan como una alternativa interpretable y como contexto metodológico, pero no se usan para decidir el ganador.

Artefactos generados:
- `llm_features.parquet`
- `llm_features_train.parquet`
- `llm_features_test.parquet`
- `bert_embeddings_train.parquet`
- `bert_embeddings_test.parquet`
- `bert_svd.joblib`
- `nlp_feature_extraction_metadata.json`


## 1. Configuración y rutas

Se cargan los artefactos del split temporal ya creado en fases anteriores. 

La caché es importante porque las llamadas LLM y los embeddings BERT pueden tardar bastante. Si la ejecución se interrumpe, el proceso puede continuar desde los archivos ya guardados.


In [19]:
# ruff: noqa: E402, I001
import hashlib
import json
import os
import re
import sys
import time
from collections.abc import Callable
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from typing import Any

import joblib
import numpy as np
import pandas as pd
from loguru import logger
from scipy.stats import kruskal
from sklearn.decomposition import TruncatedSVD


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src").exists():
            return candidate
    raise RuntimeError("No se pudo localizar la raíz del proyecto")


_DISCOVERED_PROJECT_ROOT = find_project_root(Path.cwd().resolve())
SRC_DIR = _DISCOVERED_PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from triaje_ia.config import DATA_INTERIM, DATA_PROCESSED, PROJECT_ROOT
from triaje_ia.data.cleaner import cargar_dataset_limpio

assert PROJECT_ROOT == _DISCOVERED_PROJECT_ROOT

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

logger.remove()
logger.add(sys.stderr, level="INFO")

FORCE_REBUILD = False
LLM_BACKEND = "ollama_local"
LLM_MODEL = os.getenv("OLLAMA_MODEL", "llama3.1:8b-instruct-q4_K_M ")
OLLAMA_URL = os.getenv("OLLAMA_URL", "http://localhost:11434/api/generate")
OLLAMA_MAX_WORKERS = int(os.getenv("OLLAMA_MAX_WORKERS", "4"))
OLLAMA_NUM_THREAD = int(os.getenv("OLLAMA_NUM_THREAD", "6"))
LLM_MIN_FREQ = int(os.getenv("LLM_MIN_FREQ", "5"))
LLM_MIN_ENTROPY = float(os.getenv("LLM_MIN_ENTROPY", "1.0"))
LLM_SAVE_EVERY = int(os.getenv("LLM_SAVE_EVERY", "10"))

BERT_MODEL_NAME = "emilyalsentzer/Bio_ClinicalBERT"
N_BERT_COMPONENTS = 15
BERT_BATCH_SIZE = 128

BASE_ARTIFACTS = {
    "X_train": DATA_PROCESSED / "X_train.parquet",
    "X_test": DATA_PROCESSED / "X_test.parquet",
    "y_train": DATA_PROCESSED / "y_train.parquet",
    "dataset_clean": DATA_INTERIM / "dataset_clean.parquet",
}

ARTIFACTS = {
    "llm_cache": DATA_PROCESSED / "llm_features_cache.parquet",
    "llm_full": DATA_PROCESSED / "llm_features.parquet",
    "llm_train": DATA_PROCESSED / "llm_features_train.parquet",
    "llm_test": DATA_PROCESSED / "llm_features_test.parquet",
    "bert_train": DATA_PROCESSED / "bert_embeddings_train.parquet",
    "bert_test": DATA_PROCESSED / "bert_embeddings_test.parquet",
    "bert_svd": DATA_PROCESSED / "bert_svd.joblib",
    "bert_raw_cache": DATA_PROCESSED / "bert_cls_cache",
    "metadata": DATA_PROCESSED / "nlp_feature_extraction_metadata.json",
}

for directory in [DATA_PROCESSED, ARTIFACTS["bert_raw_cache"]]:
    directory.mkdir(parents=True, exist_ok=True)

print(f"Proyecto: {PROJECT_ROOT}")
print(f"Random state: {RANDOM_STATE}")
for name, path in BASE_ARTIFACTS.items():
    print(f"{name:<12} -> {path.relative_to(PROJECT_ROOT)}")
for name, path in ARTIFACTS.items():
    print(f"{name:<12} -> {path.relative_to(PROJECT_ROOT)}")

Proyecto: C:\Users\CARLOS\triaje-ia-tfg
Random state: 42
X_train      -> data\processed\X_train.parquet
X_test       -> data\processed\X_test.parquet
y_train      -> data\processed\y_train.parquet
dataset_clean -> data\interim\dataset_clean.parquet
llm_cache    -> data\processed\llm_features_cache.parquet
llm_full     -> data\processed\llm_features.parquet
llm_train    -> data\processed\llm_features_train.parquet
llm_test     -> data\processed\llm_features_test.parquet
bert_train   -> data\processed\bert_embeddings_train.parquet
bert_test    -> data\processed\bert_embeddings_test.parquet
bert_svd     -> data\processed\bert_svd.joblib
bert_raw_cache -> data\processed\bert_cls_cache
metadata     -> data\processed\nlp_feature_extraction_metadata.json


## 2. Carga del split temporal y alineación con el texto original

El texto `chiefcomplaint` vive en el dataset limpio, mientras que los modelos trabajan con matrices `X_train` y `X_test`. Por eso se reconstruye el corte temporal y se comprueba que coincide exactamente con los artefactos ya guardados.

Estos asserts son obligatorios: si el texto no queda alineado con las filas de train/test, cualquier embedding posterior estaría asignado al paciente equivocado.

In [20]:
required = [
    BASE_ARTIFACTS["X_train"],
    BASE_ARTIFACTS["X_test"],
    BASE_ARTIFACTS["y_train"],
]
missing = [path for path in required if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Faltan artefactos base: " + ", ".join(str(p) for p in missing)
    )

X_train = pd.read_parquet(BASE_ARTIFACTS["X_train"])
X_test = pd.read_parquet(BASE_ARTIFACTS["X_test"])
y_train = pd.read_parquet(BASE_ARTIFACTS["y_train"]).squeeze()

clean_path = BASE_ARTIFACTS["dataset_clean"]
df_clean = (
    pd.read_parquet(clean_path)
    if clean_path.exists()
    else cargar_dataset_limpio(forzar=False)
)

essential_cols = {"intime", "chiefcomplaint", "acuity"}
missing_cols = essential_cols - set(df_clean.columns)
if missing_cols:
    raise ValueError(f"Faltan columnas en dataset_clean: {sorted(missing_cols)}")

df_clean = df_clean.copy()
df_clean["intime"] = pd.to_datetime(df_clean["intime"], errors="raise")
fecha_corte = df_clean["intime"].quantile(0.80)

df_train_raw = df_clean.loc[df_clean["intime"] <= fecha_corte].reset_index(drop=True)
df_test_raw = df_clean.loc[df_clean["intime"] > fecha_corte].reset_index(drop=True)

assert len(df_train_raw) == len(X_train), (len(df_train_raw), len(X_train))
assert len(df_test_raw) == len(X_test), (len(df_test_raw), len(X_test))
assert len(y_train) == len(X_train), (len(y_train), len(X_train))
assert (
    df_train_raw["acuity"].reset_index(drop=True).equals(y_train.reset_index(drop=True))
)

print(f"Fecha de corte temporal: {fecha_corte}")
print(f"Train: {len(df_train_raw):,} filas | Test: {len(df_test_raw):,} filas")
print("Distribución de acuity en train:")
print(y_train.value_counts(normalize=True).sort_index().round(4).to_string())

Fecha de corte temporal: 2180-06-06 02:38:36
Train: 334,480 filas | Test: 83,620 filas
Distribución de acuity en train:
acuity
1    0.0579
2    0.3326
3    0.5372
4    0.0695
5    0.0028


## 3. Normalización m?nima del texto

El `chiefcomplaint` de MIMIC-IV-ED suele ser muy corto, a veces solo una abreviatura. Se crean dos versiones:

- `cc_clean`: clave hacer merges.
- `cc_text_model`: versión algo más limpia para BERT, expandiendo abreviaturas frecuentes.



In [21]:
ABREVIATURAS = {
    r"\bsi\b": "suicidal ideation",
    r"\betoh\b": "alcohol intoxication",
    r"\bbrbpr\b": "rectal bleeding",
    r"\bmvc\b": "motor vehicle accident",
    r"\bs/p\b": "status post",
    r"\bn/v\b": "nausea vomiting",
    r"\babd\b": "abdominal",
    r"\bcp\b": "chest pain",
    r"\bsob\b": "shortness of breath",
    r"\baloc\b": "altered level of consciousness",
    r"\bams\b": "altered mental status",
    r"\bha\b": "headache",
    r"\bhtn\b": "hypertension",
    r"\bdm\b": "diabetes",
    r"\bdka\b": "diabetic ketoacidosis",
    r"\bpe\b": "pulmonary embolism",
    r"\buti\b": "urinary tract infection",
    r"\blle\b": "left lower extremity",
    r"\brle\b": "right lower extremity",
}


def limpiar_cc(texto: Any) -> str:
    if pd.isna(texto):
        return ""
    return str(texto).lower().strip()


def preparar_texto_modelo(texto: Any) -> str:
    if pd.isna(texto):
        return "desconocido"
    t = str(texto).lower().strip()
    for patron, expansion in ABREVIATURAS.items():
        t = re.sub(patron, expansion, t)
    t = re.sub(r"[^a-z\s]", " ", t)
    t = re.sub(r"\s+", " ", t).strip()
    return t or "desconocido"


for frame in (df_train_raw, df_test_raw):
    frame["cc_clean"] = frame["chiefcomplaint"].apply(limpiar_cc)
    frame["cc_text_model"] = frame["chiefcomplaint"].apply(preparar_texto_modelo)

assert df_train_raw["cc_clean"].notna().all()
assert df_test_raw["cc_clean"].notna().all()
assert df_train_raw["cc_text_model"].notna().all()
assert df_test_raw["cc_text_model"].notna().all()

for original, clean, model_text in (
    df_train_raw[["chiefcomplaint", "cc_clean", "cc_text_model"]]
    .head(8)
    .itertuples(index=False)
):
    print(f"{original!r} -> cc_clean={clean!r} | bert={model_text!r}")

'Abd pain, Abdominal distention' -> cc_clean='abd pain, abdominal distention' | bert='abdominal pain abdominal distention'
'Confusion, Hallucinations' -> cc_clean='confusion, hallucinations' | bert='confusion hallucinations'
'Altered mental status, B Pedal edema' -> cc_clean='altered mental status, b pedal edema' | bert='altered mental status b pedal edema'
'LEFT CHEEK SWELLING, Abscess' -> cc_clean='left cheek swelling, abscess' | bert='left cheek swelling abscess'
'L CHEEK ABSCESS' -> cc_clean='l cheek abscess' | bert='l cheek abscess'
'L FACIAL SWELLING' -> cc_clean='l facial swelling' | bert='l facial swelling'
'Suture removal' -> cc_clean='suture removal' | bert='suture removal'
'Laceration, s/p Fall' -> cc_clean='laceration, s/p fall' | bert='laceration status post fall'


## 4. Radiografía de train

Esta parte justifica por qué merece la pena trabajar con NLP. Muchos motivos de consulta son ambiguos: el mismo texto puede aparecer en distintos niveles de triaje dependiendo del contexto clínico. La entropía se calcula solo en train porque usa `acuity`.



In [22]:
def entropia(serie: pd.Series) -> float:
    p = serie.value_counts(normalize=True)
    return float(-(p * p.apply(lambda x: np.log2(x) if x > 0 else 0)).sum())


stats_cc_train = pd.DataFrame(
    {
        "freq": df_train_raw.groupby("cc_clean")["acuity"].size(),
        "entropy": df_train_raw.groupby("cc_clean")["acuity"].apply(entropia),
        "n_clases": df_train_raw.groupby("cc_clean")["acuity"].nunique(),
        "acuity_moda": df_train_raw.groupby("cc_clean")["acuity"].apply(
            lambda x: x.mode().iloc[0]
        ),
    }
).sort_values(["freq", "entropy"], ascending=[False, False])

coverage_freq5 = stats_cc_train.loc[stats_cc_train["freq"] >= 5, "freq"].sum() / len(
    df_train_raw
)
print(f"Chief complaints únicos en train: {df_train_raw['cc_clean'].nunique():,}")
print(f"Chief complaints únicos en test : {df_test_raw['cc_clean'].nunique():,}")
print(f"Cobertura train con freq >= 5  : {coverage_freq5:.1%}")
print("\nTop 15 por frecuencia en train:")
print(stats_cc_train.head(15).to_string())
print("\nTop 10 ambiguos en train (freq >= 10, entropía alta):")
print(
    stats_cc_train.query("freq >= 10")
    .sort_values("entropy", ascending=False)
    .head(10)
    .to_string()
)

Chief complaints únicos en train: 48,901
Chief complaints únicos en test : 17,205
Cobertura train con freq >= 5  : 83.2%

Top 15 por frecuencia en train:
                        freq   entropy  n_clases  acuity_moda
cc_clean                                                     
abd pain               12524  0.738488         5            3
chest pain             10739  1.129623         4            2
s/p fall                5812  1.484269         5            3
dyspnea                 5748  1.426210         4            2
headache                4126  1.092362         5            3
wound eval              4041  1.153692         5            3
si                      4011  0.191962         4            2
back pain               3879  1.305762         5            3
etoh                    3831  0.975520         5            3
fever                   2817  1.348622         4            3
altered mental status   2428  1.411617         4            2
lower back pain         2402  1.255851  

## 5. PRUEBA: flags LLM desde chief complaint

Durante la exploración se probó que un LLM que leyera cada `chiefcomplaint` y devolviera flags clínicos estructurados. Esta línea es útil para documentar que se exploró una representación semántica explicable, aunque el camino que acabó pesando más fue BERT.


In [23]:
# ruff: noqa: E501
LLM_FIELDS = [
    "severidad_estimada",
    "es_time_sensitive",
    "riesgo_deterioro",
    "sospecha_cardiaco",
    "sospecha_neurologico",
    "sospecha_infeccion_grave",
    "es_trauma_mayor",
    "es_cuadro_inespecifico",
    "complejidad_clinica",
    "patron_presentacion",
    "es_quirurgico",
    "sospecha_vascular_agudo",
    "es_exacerbacion_cronica",
    "requiere_ingreso_probable",
    "es_retorno_control",
    "riesgo_via_aerea",
    "n_quejas_mencionadas",
    "sospecha_psiquiatrico_toxicologico",
    "hemorragia_sangrado_activo",
    "sospecha_obstetrica_ginecologica",
    "dolor_extremo_implicito",
    "vulnerabilidad_inmunologica_implicita",
]

NUMERIC_DEFAULTS = {
    "severidad_estimada": 3,
    "complejidad_clinica": 1,
    "n_quejas_mencionadas": 1,
}
CATEGORICAL_DEFAULTS = {"patron_presentacion": "inespecifico"}
BOOLEAN_FIELDS = [
    f for f in LLM_FIELDS if f not in NUMERIC_DEFAULTS and f not in CATEGORICAL_DEFAULTS
]

LLM_SYSTEM_PROMPT = """
You are an experienced emergency triage nurse with 20 years of experience evaluating patients in a US emergency department.

Given a chief complaint from an emergency department record, analyze it and return a JSON object with the fields described below.

COMMON ED ABBREVIATIONS you must recognize:
- SI = suicidal ideation
- HI = homicidal ideation
- AMS = altered mental status
- SOB = shortness of breath
- MVC = motor vehicle collision
- CP = chest pain
- LOC = loss of consciousness
- ETOH = alcohol intoxication
- BRBPR = bright red blood per rectum
- N/V = nausea/vomiting
- S/P = status post
- GI = gastrointestinal
- UTI = urinary tract infection
- GSW = gunshot wound
- R/L = right/left
- ABD = abdominal
- HA = headache
- PE = pulmonary embolism
- DVT = deep vein thrombosis
- AAA = abdominal aortic aneurysm

Return these fields: severidad_estimada, es_time_sensitive, riesgo_deterioro, sospecha_cardiaco, sospecha_neurologico, sospecha_infeccion_grave, es_trauma_mayor, es_cuadro_inespecifico, complejidad_clinica, patron_presentacion, es_quirurgico, sospecha_vascular_agudo, es_exacerbacion_cronica, requiere_ingreso_probable, es_retorno_control, riesgo_via_aerea, n_quejas_mencionadas, sospecha_psiquiatrico_toxicologico, hemorragia_sangrado_activo, sospecha_obstetrica_ginecologica, dolor_extremo_implicito, vulnerabilidad_inmunologica_implicita.

Rules: use only the chief complaint text; if information is insufficient, be conservative; do not diagnose; return only valid JSON.
""".strip()


def fila_llm_default() -> dict[str, Any]:
    row = {field: False for field in BOOLEAN_FIELDS}
    row.update(NUMERIC_DEFAULTS)
    row.update(CATEGORICAL_DEFAULTS)
    return row


def parsear_respuesta_llm(texto: str) -> dict[str, Any]:
    defaults = fila_llm_default()
    match = re.search(r"\{.*\}", texto, re.DOTALL)
    if not match:
        return defaults
    try:
        payload = json.loads(match.group())
    except json.JSONDecodeError:
        return defaults

    row = defaults.copy()
    for field in BOOLEAN_FIELDS:
        row[field] = bool(payload.get(field, row[field]))
    for field, default in NUMERIC_DEFAULTS.items():
        try:
            row[field] = int(payload.get(field, default))
        except (TypeError, ValueError):
            row[field] = default
    patron = str(payload.get("patron_presentacion", "inespecifico")).lower().strip()
    row["patron_presentacion"] = (
        patron if patron in {"clasico", "atipico", "inespecifico"} else "inespecifico"
    )
    return row


def construir_llamador_llm() -> Callable[[str], dict[str, Any]]:
    import httpx

    model_name = LLM_MODEL or "llama3.1"

    def call_ollama(cc_text: str) -> dict[str, Any]:
        prompt = f"{LLM_SYSTEM_PROMPT}\n\nChief complaint: {cc_text}"
        resp = httpx.post(
            OLLAMA_URL,
            json={
                "model": model_name,
                "prompt": prompt,
                "format": "json",
                "stream": False,
                "options": {
                    "temperature": 0.0,
                    "num_predict": 300,
                    "num_ctx": 2048,
                    "num_gpu": 99,
                    "num_batch": 256,
                    "num_thread": OLLAMA_NUM_THREAD,
                },
            },
            timeout=180.0,
        )
        resp.raise_for_status()
        return parsear_respuesta_llm(resp.json()["response"])

    return call_ollama

In [24]:
def convertir_llm_features(df: pd.DataFrame) -> pd.DataFrame:
    result = df.copy()
    if "cc_clean" not in result.columns:
        raise ValueError("La tabla LLM debe contener cc_clean")
    result["cc_clean"] = result["cc_clean"].fillna("").astype(str)

    for field in BOOLEAN_FIELDS:
        if field not in result.columns:
            result[field] = False
        result[field] = result[field].fillna(False).astype(bool)
    for field, default in NUMERIC_DEFAULTS.items():
        if field not in result.columns:
            result[field] = default
        result[field] = (
            pd.to_numeric(result[field], errors="coerce")
            .fillna(default)
            .astype("int16")
        )
    if "patron_presentacion" not in result.columns:
        result["patron_presentacion"] = "inespecifico"
    result["patron_presentacion"] = (
        result["patron_presentacion"]
        .fillna("inespecifico")
        .astype(str)
        .str.lower()
        .str.strip()
    )
    result.loc[
        ~result["patron_presentacion"].isin(["clasico", "atipico", "inespecifico"]),
        "patron_presentacion",
    ] = "inespecifico"
    return result[["cc_clean", *LLM_FIELDS]]


def seleccionar_complaints_llm(stats_cc: pd.DataFrame) -> list[str]:
    corpus = stats_cc[
        (stats_cc["freq"] >= LLM_MIN_FREQ) & (stats_cc["entropy"] >= LLM_MIN_ENTROPY)
    ].copy()
    corpus = corpus.sort_values(["freq", "entropy"], ascending=[False, False])
    selected = corpus.index.fillna("").astype(str).tolist()

    cobertura_train = df_train_raw["cc_clean"].isin(selected).mean()
    cobertura_test = df_test_raw["cc_clean"].isin(selected).mean()
    logger.info(
        "Corpus LLM local: %s textos | cobertura train %.1f%% | test %.1f%%",
        f"{len(selected):,}",
        cobertura_train * 100,
        cobertura_test * 100,
    )
    return selected


def guardar_cache_llm(cache: pd.DataFrame, cache_path: Path) -> None:
    cache = convertir_llm_features(cache)
    cache.to_parquet(cache_path, index=False)


def anotar_complaints_seleccionados(
    complaints_to_process: list[str], cache_path: Path
) -> pd.DataFrame:
    if cache_path.exists():
        cache = convertir_llm_features(pd.read_parquet(cache_path))
    else:
        cache = pd.DataFrame(columns=["cc_clean", *LLM_FIELDS])

    objetivo = set(complaints_to_process)
    procesados = set(cache["cc_clean"]) if not cache.empty else set()
    pendientes = [cc for cc in complaints_to_process if cc not in procesados]
    logger.info(
        f"Cache LLM: {len(procesados & objetivo):,} procesados | "
        f"{len(pendientes):,} pendientes"
    )

    if pendientes:
        llamador = construir_llamador_llm()
        nuevas: list[dict[str, Any]] = []
        errores: list[str] = []
        inicio = time.time()

        def ejecutar(cc: str) -> dict[str, Any]:
            row = fila_llm_default()
            for intento in range(3):
                try:
                    row = llamador(cc or "no especificado")
                    break
                except Exception:
                    if intento < 2:
                        time.sleep(2)
                    else:
                        errores.append(cc)
            return {"cc_clean": cc, **row}

        with ThreadPoolExecutor(max_workers=OLLAMA_MAX_WORKERS) as pool:
            futuros = {pool.submit(ejecutar, cc): cc for cc in pendientes}
            for i, futuro in enumerate(as_completed(futuros), start=1):
                nuevas.append(futuro.result())

                if i % LLM_SAVE_EVERY == 0:
                    cache = pd.concat(
                        [cache, pd.DataFrame(nuevas)], ignore_index=True
                    ).drop_duplicates("cc_clean", keep="last")
                    guardar_cache_llm(cache, cache_path)
                    nuevas = []
                    rate = i / max(time.time() - inicio, 1e-9) * 60
                    restantes = len(pendientes) - i
                    eta_h = restantes / max(rate, 1e-9) / 60
                    logger.info(
                        f"LLM local: {i:,}/{len(pendientes):,} | "
                        f"{rate:.1f} calls/min | ETA {eta_h:.1f} h"
                    )

        if nuevas:
            cache = pd.concat(
                [cache, pd.DataFrame(nuevas)], ignore_index=True
            ).drop_duplicates("cc_clean", keep="last")
            guardar_cache_llm(cache, cache_path)

        if errores:
            logger.warning(f"LLM local: {len(errores):,} textos quedaron con defaults")

    cache = convertir_llm_features(cache)
    cache = cache[cache["cc_clean"].isin(complaints_to_process)].copy()
    assert not cache[LLM_FIELDS].isna().any().any()
    assert not cache["cc_clean"].duplicated().any()
    return cache


def fusionar_llm(cc_clean: pd.Series, cache: pd.DataFrame) -> pd.DataFrame:
    base = pd.DataFrame({"cc_clean": cc_clean.fillna("").astype(str).values})
    merged = base.merge(convertir_llm_features(cache), on="cc_clean", how="left")
    for field in BOOLEAN_FIELDS:
        merged[field] = merged[field].fillna(False).astype("int8")
    for field, default in NUMERIC_DEFAULTS.items():
        merged[field] = (
            pd.to_numeric(merged[field], errors="coerce")
            .fillna(default)
            .astype("int16")
        )
    merged["patron_presentacion"] = (
        merged["patron_presentacion"]
        .fillna("inespecifico")
        .astype(str)
        .str.lower()
        .str.strip()
    )

    dummies = pd.get_dummies(
        merged["patron_presentacion"], prefix="llm_patron", dtype="int8"
    )
    for col in ["llm_patron_clasico", "llm_patron_atipico", "llm_patron_inespecifico"]:
        if col not in dummies.columns:
            dummies[col] = np.int8(0)

    numeric = merged.drop(columns=["cc_clean", "patron_presentacion"])
    return pd.concat([numeric, dummies[sorted(dummies.columns)]], axis=1)


ccs_llm = seleccionar_complaints_llm(stats_cc_train)
llm_outputs_exist = all(
    ARTIFACTS[name].exists()
    for name in ["llm_full", "llm_train", "llm_test", "llm_cache"]
)
llm_metadata_match = False
if ARTIFACTS["metadata"].exists():
    previous_metadata = json.loads(ARTIFACTS["metadata"].read_text(encoding="utf-8"))
    previous_llm = previous_metadata.get("llm", {})
    llm_metadata_match = (
        previous_llm.get("backend") == LLM_BACKEND
        and previous_llm.get("model") == LLM_MODEL
    )

if llm_outputs_exist and llm_metadata_match and not FORCE_REBUILD:
    llm_cache = convertir_llm_features(pd.read_parquet(ARTIFACTS["llm_cache"]))
    llm_train = pd.read_parquet(ARTIFACTS["llm_train"])
    llm_test = pd.read_parquet(ARTIFACTS["llm_test"])
    llm_full = pd.read_parquet(ARTIFACTS["llm_full"])
else:
    llm_cache = anotar_complaints_seleccionados(ccs_llm, ARTIFACTS["llm_cache"])
    llm_train = fusionar_llm(df_train_raw["cc_clean"], llm_cache)
    llm_test = fusionar_llm(df_test_raw["cc_clean"], llm_cache)
    llm_full = pd.concat(
        [
            pd.concat(
                [
                    pd.DataFrame(
                        {
                            "split": "train",
                            "row_in_split": np.arange(len(llm_train)),
                            "cc_clean": df_train_raw["cc_clean"].values,
                        }
                    ),
                    llm_train.reset_index(drop=True),
                ],
                axis=1,
            ),
            pd.concat(
                [
                    pd.DataFrame(
                        {
                            "split": "test",
                            "row_in_split": np.arange(len(llm_test)),
                            "cc_clean": df_test_raw["cc_clean"].values,
                        }
                    ),
                    llm_test.reset_index(drop=True),
                ],
                axis=1,
            ),
        ],
        ignore_index=True,
    )
    llm_cache.to_parquet(ARTIFACTS["llm_cache"], index=False)
    llm_train.to_parquet(ARTIFACTS["llm_train"], index=False)
    llm_test.to_parquet(ARTIFACTS["llm_test"], index=False)
    llm_full.to_parquet(ARTIFACTS["llm_full"], index=False)

assert len(llm_train) == len(X_train)
assert len(llm_test) == len(X_test)
assert "acuity" not in llm_train.columns
assert "acuity" not in llm_test.columns
assert not llm_train.isna().any().any()
assert not llm_test.isna().any().any()
print(f"LLM cache : {llm_cache.shape}")
print(f"LLM train : {llm_train.shape}")
print(f"LLM test  : {llm_test.shape}")

2026-05-21 17:37:24.913 | INFO     | __main__:seleccionar_complaints_llm:44 - Corpus LLM local: %s textos | cobertura train %.1f%% | test %.1f%%
2026-05-21 17:37:24.915 | INFO     | __main__:anotar_complaints_seleccionados:69 - Cache LLM: 0 procesados | 1,729 pendientes
C:\Users\CARLOS\AppData\Local\Temp\ipykernel_40300\650927318.py:10: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  result[field] = result[field].fillna(False).astype(bool)
C:\Users\CARLOS\AppData\Local\Temp\ipykernel_40300\650927318.py:10: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  resu

LLM cache : (1729, 23)
LLM train : (334480, 24)
LLM test  : (83620, 24)


## 6. Comprobación descriptiva de las features LLM en train

Esta comprobación solo busca confirmar que las variables no son constantes y que tienen alguna señal en train. No se decide aqué si entran o no en el modelo final.

In [25]:
def kruskal_por_feature(features: pd.DataFrame, y: pd.Series) -> pd.DataFrame:
    rows = []
    y_values = y.reset_index(drop=True)
    for col in features.columns:
        values = features[col].reset_index(drop=True)
        if values.nunique(dropna=False) < 2:
            continue
        try:
            grupos = [
                values[y_values == k].astype(float).values
                for k in sorted(y_values.unique())
            ]
            h_stat, p_value = kruskal(*grupos)
        except ValueError:
            continue
        rows.append({"feature": col, "H": float(h_stat), "p_value": float(p_value)})
    return pd.DataFrame(rows).sort_values("H", ascending=False).reset_index(drop=True)


llm_stats = kruskal_por_feature(llm_train, y_train)
assert not llm_stats.empty
print(llm_stats.head(20).round(4).to_string(index=False))

                              feature          H  p_value
                    es_time_sensitive 14900.4800   0.0000
              dolor_extremo_implicito 13983.0523   0.0000
                   es_retorno_control 13345.5649   0.0000
            requiere_ingreso_probable 13085.8264   0.0000
                     riesgo_deterioro  8555.8893   0.0000
              es_exacerbacion_cronica  7151.5349   0.0000
                    sospecha_cardiaco  6940.2051   0.0000
              sospecha_vascular_agudo  6757.6131   0.0000
                      es_trauma_mayor  5962.5335   0.0000
               es_cuadro_inespecifico  5852.2085   0.0000
                 sospecha_neurologico  4210.3614   0.0000
                        es_quirurgico  2966.4857   0.0000
                     riesgo_via_aerea  2831.4618   0.0000
           hemorragia_sangrado_activo  1273.4432   0.0000
   sospecha_psiquiatrico_toxicologico   473.2547   0.0000
             sospecha_infeccion_grave   321.7348   0.0000
vulnerabilidad

## 7. Línea principal: Bio_ClinicalBERT + SVD

La línea que se conserva con más peso para el pipeline final es Bio_ClinicalBERT. La motivación es que los chief complaints son textos muy cortos y llenos de abreviaturas, por lo que una representación semántica clínica puede capturar similitudes que las regex o los flags manuales no ven.

El procedimiento es:

1. Obtener el vector CLS de Bio_ClinicalBERT para cada texto normalizado.
2. Cachear los CLS por lotes para poder reanudar.
3. Ajustar `TruncatedSVD` solo en train.
4. Transformar train y test con ese mismo SVD.
5. Guardar 15 componentes `bert_svd_00` ... `bert_svd_14`.



In [26]:
def cargar_o_calcular_cls(
    textos: list[str], cache_dir: Path, batch_size: int = BERT_BATCH_SIZE
) -> np.ndarray:
    cache_dir.mkdir(parents=True, exist_ok=True)
    n_batches = (len(textos) + batch_size - 1) // batch_size
    batch_paths = [cache_dir / f"batch_{i:05d}.parquet" for i in range(n_batches)]

    if all(path.exists() for path in batch_paths):
        frames = [pd.read_parquet(path) for path in batch_paths]
        return pd.concat(frames, ignore_index=True).to_numpy(dtype=np.float32)

    import torch
    from transformers import AutoModel, AutoTokenizer

    tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL_NAME)
    modelo = AutoModel.from_pretrained(BERT_MODEL_NAME)
    modelo.eval()
    device = "cuda" if torch.cuda.is_available() else "cpu"
    modelo = modelo.to(device)
    print(f"Bio_ClinicalBERT cargado en: {device}")

    for batch_idx, start in enumerate(range(0, len(textos), batch_size)):
        path = batch_paths[batch_idx]
        if path.exists():
            continue
        batch = textos[start : start + batch_size]
        enc = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=32,
            return_tensors="pt",
        ).to(device)
        with torch.no_grad():
            out = modelo(**enc)
        cls = out.last_hidden_state[:, 0, :].cpu().numpy().astype(np.float32)
        cols = [f"cls_{i:03d}" for i in range(cls.shape[1])]
        pd.DataFrame(cls, columns=cols).to_parquet(path, index=False)
        logger.info(f"BERT CLS cacheado: lote {batch_idx + 1}/{n_batches}")

    frames = [pd.read_parquet(path) for path in batch_paths]
    return pd.concat(frames, ignore_index=True).to_numpy(dtype=np.float32)


def construir_bert_svd() -> tuple[pd.DataFrame, pd.DataFrame, TruncatedSVD]:
    if (
        all(
            path.exists()
            for path in [
                ARTIFACTS["bert_train"],
                ARTIFACTS["bert_test"],
                ARTIFACTS["bert_svd"],
            ]
        )
        and not FORCE_REBUILD
    ):
        return (
            pd.read_parquet(ARTIFACTS["bert_train"]),
            pd.read_parquet(ARTIFACTS["bert_test"]),
            joblib.load(ARTIFACTS["bert_svd"]),
        )

    cls_train = cargar_o_calcular_cls(
        df_train_raw["cc_text_model"].tolist(), ARTIFACTS["bert_raw_cache"] / "train"
    )
    cls_test = cargar_o_calcular_cls(
        df_test_raw["cc_text_model"].tolist(), ARTIFACTS["bert_raw_cache"] / "test"
    )
    assert cls_train.shape[0] == len(df_train_raw)
    assert cls_test.shape[0] == len(df_test_raw)

    svd = TruncatedSVD(n_components=N_BERT_COMPONENTS, random_state=RANDOM_STATE)
    # El SVD aprende la proyección solo con train; test se transforma con ese ajuste.
    bert_train_arr = svd.fit_transform(cls_train)
    bert_test_arr = svd.transform(cls_test)

    cols = [f"bert_svd_{i:02d}" for i in range(N_BERT_COMPONENTS)]
    bert_train_df = pd.DataFrame(bert_train_arr, columns=cols)
    bert_test_df = pd.DataFrame(bert_test_arr, columns=cols)

    bert_train_df.to_parquet(ARTIFACTS["bert_train"], index=False)
    bert_test_df.to_parquet(ARTIFACTS["bert_test"], index=False)
    joblib.dump(svd, ARTIFACTS["bert_svd"], compress=3)
    return bert_train_df, bert_test_df, svd


bert_train, bert_test, bert_svd = construir_bert_svd()
expected_cols = [f"bert_svd_{i:02d}" for i in range(N_BERT_COMPONENTS)]
assert list(bert_train.columns) == expected_cols
assert list(bert_test.columns) == expected_cols
assert len(bert_train) == len(X_train)
assert len(bert_test) == len(X_test)
assert not bert_train.isna().any().any()
assert not bert_test.isna().any().any()
assert bert_svd.n_components == N_BERT_COMPONENTS

print(f"BERT train: {bert_train.shape}")
print(f"BERT test : {bert_test.shape}")
print(
    f"Varianza explicada por SVD train: {bert_svd.explained_variance_ratio_.sum():.2%}"
)

BERT train: (334480, 15)
BERT test : (83620, 15)
Varianza explicada por SVD train: 68.34%


## 8. Señal descriptiva de los componentes BERT en train

Para cerrar la extracción, se comprueba si los componentes BERT-SVD tienen variabilidad y relación con el nivel de triaje dentro de train. 

In [27]:
bert_stats = kruskal_por_feature(bert_train, y_train)
assert len(bert_stats) == N_BERT_COMPONENTS
print(bert_stats.round(4).to_string(index=False))

    feature          H  p_value
bert_svd_05 17734.0660      0.0
bert_svd_03 14639.8658      0.0
bert_svd_01 11303.1838      0.0
bert_svd_13  8433.6069      0.0
bert_svd_00  7857.9107      0.0
bert_svd_08  6363.7457      0.0
bert_svd_02  6132.8710      0.0
bert_svd_07  5151.2295      0.0
bert_svd_06  3825.1922      0.0
bert_svd_04  3610.0621      0.0
bert_svd_12  3532.4527      0.0
bert_svd_14  3190.7175      0.0
bert_svd_09  3130.4739      0.0
bert_svd_10  1017.2743      0.0
bert_svd_11   690.8982      0.0


## 9. Metadata de los artefactos

Se guarda una metadata sencilla con shapes, columnas, configuración de BERT y hash del prompt LLM. Esto permite que los notebooks posteriores comprueben que están usando la misma extracción.

In [28]:
def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


metadata = {
    "random_state": RANDOM_STATE,
    "fecha_corte": str(fecha_corte),
    "n_train": int(len(X_train)),
    "n_test": int(len(X_test)),
    "llm": {
        "backend": LLM_BACKEND,
        "model": LLM_MODEL,
        "ollama_url": OLLAMA_URL,
        "max_workers": OLLAMA_MAX_WORKERS,
        "num_thread": OLLAMA_NUM_THREAD,
        "selection": {
            "min_freq_train": LLM_MIN_FREQ,
            "min_entropy_train": LLM_MIN_ENTROPY,
            "n_selected": len(ccs_llm),
        },
        "fields": LLM_FIELDS,
        "prompt_sha256": hashlib.sha256(LLM_SYSTEM_PROMPT.encode("utf-8")).hexdigest(),
        "shape_train": list(llm_train.shape),
        "shape_test": list(llm_test.shape),
    },
    "bert": {
        "model_name": BERT_MODEL_NAME,
        "n_components": N_BERT_COMPONENTS,
        "columns": list(bert_train.columns),
        "explained_variance_ratio_sum": float(bert_svd.explained_variance_ratio_.sum()),
        "shape_train": list(bert_train.shape),
        "shape_test": list(bert_test.shape),
    },
    "artifacts": {
        name: str(path.relative_to(PROJECT_ROOT))
        for name, path in ARTIFACTS.items()
        if name != "bert_raw_cache"
    },
    "hashes": {
        "llm_features": sha256_file(ARTIFACTS["llm_full"]),
        "bert_train": sha256_file(ARTIFACTS["bert_train"]),
        "bert_test": sha256_file(ARTIFACTS["bert_test"]),
        "bert_svd": sha256_file(ARTIFACTS["bert_svd"]),
    },
}

ARTIFACTS["metadata"].write_text(
    json.dumps(metadata, indent=2, ensure_ascii=False), encoding="utf-8"
)
print(f"Metadata guardada en: {ARTIFACTS['metadata'].relative_to(PROJECT_ROOT)}")
print(
    json.dumps(
        {
            "llm_train": metadata["llm"]["shape_train"],
            "bert_train": metadata["bert"]["shape_train"],
            "bert_variance": round(metadata["bert"]["explained_variance_ratio_sum"], 4),
        },
        indent=2,
    )
)

Metadata guardada en: data\processed\nlp_feature_extraction_metadata.json
{
  "llm_train": [
    334480,
    24
  ],
  "bert_train": [
    334480,
    15
  ],
  "bert_variance": 0.6834
}


## 10. Cierre

Este notebook deja preparados los artefactos NLP sin tomar decisiones con el test. La parte LLM queda documentada como representación interpretable explorada. La parte BERT-SVD queda como representación semántica principal para evaluar en el notebook de entrenamiento.

El siguiente paso es `07_model_training.ipynb`, donde estas variables se compararán mediante validación cruzada y OOF dentro de train.